# Transfer learning with an EO foundation model for oil spill detection in a data-scarce setting

This tutorial demonstrates how to use a globally pretrained Earth Observation (EO) foundation model to perform image‑level classification on Sentinel‑1 SAR data.

**Problem statement:**  
Given a Sentinel‑1 SAR image in Sigma0 backscatter (VV and VH polarizations), expressed in decibel (dB) units, the task is to determine whether the image:
- contains an oil spill (*class = oil*),
- represents clean water (*class = clean*) or contains an oil look‑alike phenomenon (*class = lookalike*).
  
**Constraint:** We only have about 100 labelled images of each class to start with. This problem domain is known as data-scarce setting or data-limited regime.

**Proposed approach:**  
We build a supervised classifier on top of a pretrained EO foundation model and adapt it to this task by training a lightweight classification head using labeled SAR imagery.

**Model choice:**  
In this tutorial, we use the SATLAS pretrained model from the Allen Institute, which provides globally trained representations for Earth observation data. See the official repository for details on the model architecture, training data, and licensing:
https://github.com/allenai/satlaspretrain_models and https://huggingface.co/allenai/satlas-pretrain

**Dataset:**  
The dataset used in this tutorial is obtained from Zenodo (DOI: 10.5281/zenodo.8346859). Please refer to the original source (https://zenodo.org/records/8346860) for licensing terms and usage restrictions. Note the dataset is larger but I chose a small random subset of the train dataset for the purpose of this tutorial.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

In [ ]:
import OilSpillClassification as oilcls

In [ ]:
train_images_dir_oil = r"C:\Users\user\Documents\oilspill\dataset\train\oil"
train_images_dir_look= r"C:\Users\user\Documents\oilspill\dataset\train\lookalike"
train_images_dir_clean = r"C:\Users\user\Documents\oilspill\dataset\train\clean"
train_df = {'images_dir_oil': train_images_dir_oil,
            'images_dir_lookalike': train_images_dir_look,
            'images_dir_clean': train_images_dir_clean}

val_images_dir_oil = r"C:\Users\user\Documents\oilspill\dataset\val\oil"
val_images_dir_look = r"C:\Users\user\Documents\oilspill\dataset\val\lookalike"
val_images_dir_clean = r"C:\Users\user\Documents\oilspill\dataset\val\clean"
val_df = {'images_dir_oil': val_images_dir_oil,
            'images_dir_lookalike': val_images_dir_look,
            'images_dir_clean': val_images_dir_clean}

test_images_dir_oil = r"C:\Users\user\Documents\oilspill\dataset\test\oil"
test_images_dir_look = r"C:\Users\user\Documents\oilspill\dataset\test\lookalike"
test_images_dir_clean = r"C:\Users\user\Documents\oilspill\dataset\test\clean"
test_df = {'images_dir_oil': test_images_dir_oil,
          'images_dir_lookalike': test_images_dir_look,
          'images_dir_clean': test_images_dir_clean}

input_size = 512 # I choose to downsize images but we can experiment with original resolutions (2048 pixels) too.

# checkpoints will be stored here.
out_dir_model = r"C:\Users\user\Documents\oilspill\models"
# run logs (tensorboard) will be stored here.
out_dir_runs = r"C:\Users\user\Documents\oilspill\runs"
#If your training interrupts and you want to resume training.
checkpoint_to_resume = r"C:\user\Documents\oilspill\models\best_model_satlas.pt" 
#once trained, this should represent the model with best F1-score on validation data
checkpoint_best = r"C:\Users\user\Documents\oilspill\models\best_model_satlas.pt" 

# Here I choose to merge clean & lookalike classes (binary classification) but we can of course experiment with three classes too.

# Settings for binary classification
class_names = ["clean","oil"]
num_classes = 2

# # Settings for three-way classification
# class_names = ["clean","oil", "lookalike"]
# num_classes = 3

## Start training
While training, the logs will be stored via tensorboard for experiment tracking. Use "tensorboard --logdir" to review the expeirment.

The progress will also be saved as plots.

The checkpoints will be saved after every "save_model_every" epochs, and the best model (based on validation F1-score) at each epoch will be saved too, if achieved.

In [ ]:
run_name = oilcls.main_train(train_df=train_df,
                            val_df=val_df,
                            test_df=test_df,
                            ckpt_dir=out_dir_model,
                            run_dir=out_dir_runs,
                            model_type = "satlas",
                            augment_train=True,
                            num_epochs=21,
                            batch_size=64,
                            lr=1e-3,
                            weight_decay=1e-4,
                            freeze_backbone=True,
                            freeze_backbone_partial=False,
                            use_scheduler=True,
                            save_model_every=5,
                            resume_training=False,
                            resume_weights_only=False,
                            resume_ckpt_path=checkpoint_to_resume,
                            input_size=input_size,
                            num_classes=num_classes,
                            class_labels=class_names)

## Test the performance
Classification metrics from the best trained model will be calculated and printed.

Confusion matrices will be stored.

In [ ]:
out_dir_plots = os.path.join(out_dir_runs, "plots_" + run_name)
os.makedirs(out_dir_plots, exist_ok=True)
oilcls.main_test(train_df=train_df,
            val_df=val_df,
            test_df=test_df,
            model_path=checkpoint_best,
            plot_out_dir=out_dir_plots,
            model_type = "satlas",
            batch_size=64,
            input_size=input_size,
            num_classes=num_classes,
            class_labels=class_names)

## Infer
Note that although I am inferring examples from the test dataset, the labels are not being used.

The outputs from the inference include:

- Prediction CSV: For each image, the predicted class will be indicated along with the scores.
- CAM: For each image, a classification activation map geo-referenced with the same transform/crs as the input image will be saved.

In [ ]:
oilcls.main_infer(images_dir=test_images_dir_oil, model_path=checkpoint_best,
               out_masks_dir=os.path.join(out_dir_runs, "infer_oil"), model_type="satlas",
               batch_size=8, input_size=input_size, num_classes=num_classes, class_labels=class_names)
oilcls.main_infer(images_dir=test_images_dir_look, model_path=checkpoint_best,
           out_masks_dir=os.path.join(out_dir_runs, "infer_lookalike"), model_type="satlas",
           batch_size=8, input_size=input_size, num_classes=num_classes, class_labels=class_names)
oilcls.main_infer(images_dir=test_images_dir_clean, model_path=checkpoint_best,
           out_masks_dir=os.path.join(out_dir_runs, "infer_clean"), model_type="satlas",
           batch_size=8, input_size=input_size, num_classes=num_classes, class_labels=class_names)

### Here are samples of plots and results produced by this model.

Sample prediction csv content created during inference:

    image_path,predicted_class,score_clean,score_oil
    C:\Users\user\Documents\oilspill\dataset\test\oil\00000.tif,oil,0.33203956,0.66796046
    C:\Users\user\Documents\oilspill\dataset\test\oil\00001.tif,oil,0.2085741,0.7914259
    C:\Users\user\Documents\oilspill\dataset\test\oil\00002.tif,oil,0.30676997,0.69323003
    C:\Users\user\Documents\oilspill\dataset\test\oil\00003.tif,oil,0.0797964,0.92020357


Sample CAM created during inference: 

![](./Assets/cam_presentation_2.png)


Training progress captured by Tensorboard:

![](./Assets/train_acc.png)

![](./Assets/train_loss.png)

Performance evaluation done via main_test function:

    -------------------- RESULTS ON TRAIN DATASET
    Overall Accuracy: 0.8851963746223565
    Classification report
                  precision    recall  f1-score   support
    
           clean       0.94      0.84      0.89       176
             oil       0.83      0.94      0.88       155
    
        accuracy                           0.89       331
       macro avg       0.89      0.89      0.89       331
    weighted avg       0.89      0.89      0.89       331
    
![](./Assets/CM_TRAIN.png)
    
    -------------------- RESULTS ON VALIDATION DATASET
    Overall Accuracy: 0.9433962264150944
    Classification report
                  precision    recall  f1-score   support
    
           clean       0.96      0.93      0.95        28
             oil       0.92      0.96      0.94        25
    
        accuracy                           0.94        53
       macro avg       0.94      0.94      0.94        53
    weighted avg       0.94      0.94      0.94        53

![](./Assets/CM_VALIDATION.png)

    -------------------- RESULTS ON TEST DATASET
    Overall Accuracy: 0.8266666666666667
    Classification report
                  precision    recall  f1-score   support
    
           clean       0.88      0.86      0.87       300
             oil       0.73      0.76      0.75       150
    
        accuracy                           0.83       450
       macro avg       0.80      0.81      0.81       450
    weighted avg       0.83      0.83      0.83       450

![](./Assets/CM_TEST.png)

## Exercise 1

Visualize misclassified samples geographically and identify spatial patterns associated with errors.

How to do this?

- Filter misclassified samples
- Plot them on the map
- Compare error locations with:
    - coastlines
    - regions with dense shipping
    - regions absent from training data
- Divide them by their original lookalike and clean labels, and see what percentage of the errors is caused by confusions with lookalike cases?
- Investigate the lookalike dataset, do you see room for further curation?
- Try the three-way classification and see whether the confusions decrease.


## Exercise 2

Compare the same experiments but with a RESNET18 model pretrained on ImageNet dataset.
This time, unfreeze the backbone as we know a frozen backbone is not going to work well.